# Week 4 assignment

Tehtävän tavoitteena on oppia hyödyntämään valmiiksi opetettuja sanaupotuksia (word embeddings) ja syventää ymmärrystä siitä, miten kieli muunnetaan matemaattisiksi sanavektoreiksi."

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import tensorflow_datasets as tfds
import numpy as np
import time
from google.colab import drive

drive.mount('/content/drive') # Google drive
path = '/content/drive/MyDrive/Resources' # Dataset location

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Tämä koodiblokki toteuttaa datan esikäsittelyvaiheen, jossa raakatekstimuotoinen GloVe-aineisto muunnetaan tietokoneen ymmärtämäksi hakurakenteeksi. Koodissa suodatetaan aineistosta virheelliset rivit ja haetaan kohdesanojen koordinaatit 200-ulotteisessa vektoriavaruudessa

In [ ]:
from os import linesep
glove_path = '/content/drive/MyDrive/Resources/glove.twitter.27B.200d.txt'
embeddings_index = {}
DIM = 200

with open(glove_path, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.rstrip().split()

        if len(values) < DIM + 1:
            continue

        word = values[0]

        try:
            vector = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = vector
        except ValueError:
            continue

word1, word2, word3 = 'man', 'woman', 'king'

v_man = embeddings_index.get(word1.lower().strip())
v_woman = embeddings_index.get(word2.lower().strip())
v_king = embeddings_index.get(word3.lower().strip())

print(f"Sanakirjan koko: {len(embeddings_index)}")

KeyboardInterrupt: 

Tässä tarkastetaan, onko haetut sanat löytynynyt sanakirjasta.

In [ ]:
missing = []

if v_man is None: missing.append(word1)
if v_woman is None: missing.append(word2)
if v_king is None: missing.append(word3)

if missing:
    print(f"VIRHE: Seuraavia sanoja ei löytynyt sanakirjasta: {missing}")
    print(list(embeddings_index.keys())[:5])
else:
    target_vector = v_woman - v_man + v_king
    print("Laskutoimitus onnistui!")

Laskutoimitus onnistui!


Tämän funktion tarkoituksena on löytää sanakirjasta se sana, joka on matemaattisesti lähimpänä laskettua tulosvektoria.

In [ ]:
def find_closest_improved(target_vec, embeddings, exclude_words):
    best_word = None
    max_sim = -1.0

    target_norm = np.linalg.norm(target_vec)

    for word, vec in embeddings.items():
        if word in exclude_words:
            continue

        dot_product = np.dot(target_vec, vec)
        sim = dot_product / (target_norm * np.linalg.norm(vec))

        if sim > max_sim:
            max_sim = sim
            best_word = word

    return best_word, max_sim

result, score = find_closest_improved(target_vector, embeddings_index, ['man', 'woman', 'king'])

print(f"\nTehtävän tulos:")
print(f"Lähin sana: {result}")
print(f"Samankaltaisuus: {score:.4f}")


Tehtävän tulos:
Lähin sana: queen
Samankaltaisuus: 0.6560


In [ ]:

def cosine_sim(a, b, eps=1e-10):
    return np.dot(a, b) / ((np.linalg.norm(a) * np.linalg.norm(b)) + eps)

def find_topk_closest(target_vec, embeddings, k=10, exclude_words=None):
    if exclude_words is None:
        exclude_words = set()
    else:
        exclude_words = set(exclude_words)

    results = []
    target_dim = target_vec.shape[0]
    target_norm = np.linalg.norm(target_vec)

    if target_norm == 0:
        raise ValueError("target_vec norm is 0")

    for word, vec in embeddings.items():
        if word in exclude_words:
            continue
        if vec.shape[0] != target_dim:
            continue

        vec_norm = np.linalg.norm(vec)
        if vec_norm == 0:
            continue

        sim = np.dot(target_vec, vec) / (target_norm * vec_norm)
        results.append((word, float(sim)))

    results.sort(key=lambda x: x[1], reverse=True)
    return results[:k]

target_vector = v_woman - v_man + v_king

top = find_topk_closest(
    target_vector,
    embeddings_index,
    k=10,
    exclude_words=["man", "woman", "king"]
)

for i, (w, s) in enumerate(top, 1):
    print(f"{i:2d}. {w:15s}  cos={s:.4f}")

 1. queen            cos=0.6560
 2. prince           cos=0.5545
 3. princess         cos=0.5415
 4. royal            cos=0.5323
 5. mother           cos=0.5083
 6. elizabeth        cos=0.5036
 7. women            cos=0.4758
 8. lion             cos=0.4730
 9. lady             cos=0.4718
10. wife             cos=0.4624
